# 🟥 Notebook 3: Real Pub/Sub with Redis

In notebook 1 we built a pub/sub bus from scratch in one Python process. Now let's do the same thing across processes (or even machines) using **Redis Pub/Sub**.

Redis is a tiny, fast in-memory data store. It also happens to ship with a pub/sub feature: clients `SUBSCRIBE` to channels, and `PUBLISH` sends a message to everyone subscribed.

## Learning objectives
- Run Redis with Docker Compose.
- Use the `redis-py` client to publish and subscribe.
- Notice what Redis Pub/Sub does **not** give you (durability, replay).

## 🛠️ Setup

Start Redis:

```bash
cd 01-foundations/messaging-basics
docker compose up -d
```

You can poke at Redis from another terminal with `docker exec -it messaging-basics-redis-1 redis-cli` if you like.

Install Python deps:
```bash
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't show up.

In [ ]:
import redis
import threading
import time

# Connect to the Redis we started with docker compose
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("ping:", r.ping())   # should print True

In [ ]:
# Subscriber runs in a background thread. In a real app this would be a
# separate process or service.
received = []

def subscriber():
    pubsub = r.pubsub()
    pubsub.subscribe("orders")
    for message in pubsub.listen():
        if message["type"] != "message":
            continue            # skip subscribe-confirmation messages
        received.append(message["data"])
        if message["data"] == "STOP":
            return

t = threading.Thread(target=subscriber, daemon=True)
t.start()

time.sleep(0.2)  # give the subscriber a moment to attach
for i in range(3):
    r.publish("orders", f"order-{i}")
r.publish("orders", "STOP")
t.join(timeout=2)

print("subscriber received:", received)

## ⚠️ Redis Pub/Sub is **fire-and-forget**

Redis Pub/Sub is at-most-once delivery: if a subscriber is offline when you publish, it **never sees that message** — there is no replay. There is also no acknowledgment.

Try this experiment yourself: publish a message *before* starting a subscriber and confirm the subscriber sees nothing.

In [ ]:
# No subscribers attached right now.
r.publish("orders", "you-will-never-see-me")

received2 = []
def subscriber2():
    pubsub = r.pubsub()
    pubsub.subscribe("orders")
    for message in pubsub.listen():
        if message["type"] != "message":
            continue
        received2.append(message["data"])
        return

t = threading.Thread(target=subscriber2, daemon=True)
t.start()
time.sleep(0.2)
r.publish("orders", "but-you-will-see-this")
t.join(timeout=2)

print("late subscriber received:", received2)

## ✅ Recap

- Redis Pub/Sub is great for low-latency *real-time* fan-out (live dashboards, chat, presence).
- It is **not** a durable message broker. Use Kafka, NATS JetStream, or Redis Streams when you need replay and persistence.
- The mental model is identical to the in-memory bus from notebook 1 — just over the network.